In [15]:
import pandas as pd
import numpy as np
import random
import copy
import math
from datetime import date
from datetime import datetime
from geopy.distance import geodesic
import warnings


warnings.filterwarnings('ignore')

In [2]:
wkday_df=pd.read_excel('reviewtool_20241223_VTA_RouteLevelComparison(Wkday & WkEnd)_Latest_01.xlsx',sheet_name='WkDAY Route Comparison')
wkday_dir_df=pd.read_excel('reviewtool_20241223_VTA_RouteLevelComparison(Wkday & WkEnd)_Latest_01.xlsx',sheet_name='WkDAY Route DIR Comparison')
wkend_df=pd.read_excel('reviewtool_20241223_VTA_RouteLevelComparison(Wkday & WkEnd)_Latest_01.xlsx',sheet_name='WkEND Route Comparison')
wkend_dir_df=pd.read_excel('reviewtool_20241223_VTA_RouteLevelComparison(Wkday & WkEnd)_Latest_01.xlsx',sheet_name='WkEND Route DIR Comparison')

In [17]:
wkend_time_df=pd.read_excel('reviewtool_20241224_VTA_RouteLevelComparison(Wkday & WkEnd)_Latest_01.xlsx',sheet_name='WkEND Time Data')

In [18]:
wkend_time_df.columns

Index(['Original Text', 0, 1, 2, 3, 4, 5, 'Time Range', 'Display_Text'], dtype='object')

In [19]:
wkend_time_df[['Display_Text','Original Text','Time Range',0, 1, 2, 3, 4, 5]]

,Display_Text,Original Text,Time Range,0,1,2,3,4,5
0,1,AM1,Before 5:00 am,0,0,0,0,0,0
1,2,AM2,5:00 am - 6:00 am,0,0,0,0,0,0
2,3,AM3,6:00 am - 7:00 am,0,0,4,0,0,0
3,4,MID1,7:00 am - 8:00 am,0,0,0,9,0,0
4,5,MID2,8:00 am - 9:00 am,0,0,0,5,0,0
5,6,MID7,9:00 am - 10:00 am,0,0,0,6,0,0
6,7,MID3,10:00 am - 11:00 am,0,0,0,0,19,0
7,8,MID4,11:00 am - 12:00 pm,0,0,0,0,11,0
8,9,MID5,12:00 pm - 1:00 pm,0,0,0,0,8,0
9,10,MID6,1:00 pm - 2:00 pm,0,0,0,0,10,0


In [3]:
detail_df=pd.read_excel('details_vta_CA_od_excel.xlsx',sheet_name='TOD')
detail_df.head()

,OPPO_TIME[CODE],TIME_ON[Code],TIME_ON,TIME_PERIOD[Code],TIME_PERIOD,START_TIME,AGENCY,WKEND_TIME_PERIOD[Code],WKEND_TIME_PERIOD,Unnamed: 9,...,Unnamed: 11,Unnamed: 12,BOARD_TIMEPERIOD[Code],BOARD_TIMEPERIOD,TIME_PERIOD[Code]2,TIME_PERIOD2,Unnamed: 17,Unnamed: 18,Unnamed: 19,Unnamed: 20
0,AM1,AM1,Before 5:00 am,0,OWL,01:00:00,NaN,NaN,NaN,NaN,...,NaN,NaN,AM1,12:00 am - 4:59 am,0.0,OWL,NaN,AM1 = 12:00 am - 4:59 am,NaN,0 = OWL
1,AM2,AM2,5:00 am - 6:00 am,1,EARLY AM,05:00:00,NaN,NaN,NaN,NaN,...,NaN,NaN,AM2,5:00 am - 5:59 am,1.0,EARLY AM,NaN,AM2 = 5:00 am - 5:59 am,NaN,1 = EARLY AM
2,AM3,AM3,6:00 am - 7:00 am,2,AM,06:00:00,NaN,NaN,NaN,NaN,...,NaN,NaN,AM3,6:00 am - 9:59 am,2.0,AM,NaN,AM3 = 6:00 am - 9:59 am,NaN,2 = AM
3,MID1,MID1,7:00 am - 8:00 am,2,AM,07:00:00,NaN,NaN,NaN,NaN,...,NaN,NaN,MID,10:00 am - 2:59 pm,3.0,MID,NaN,MID = 10:00 am - 2:59 pm,NaN,3 = MID
4,MID2,MID2,8:00 am - 9:00 am,2,AM,08:00:00,NaN,NaN,NaN,NaN,...,NaN,NaN,PM1,3:00 pm - 6:59 pm,4.0,PM,NaN,PM1 = 3:00 pm - 6:59 pm,NaN,4 = PM


In [4]:
detail_df.columns

Index(['OPPO_TIME[CODE]', 'TIME_ON[Code]', 'TIME_ON', 'TIME_PERIOD[Code]',
       'TIME_PERIOD', 'START_TIME', 'AGENCY', 'WKEND_TIME_PERIOD[Code]',
       'WKEND_TIME_PERIOD', 'Unnamed: 9', 'Unnamed: 10', 'Unnamed: 11',
       'Unnamed: 12', 'BOARD_TIMEPERIOD[Code]', 'BOARD_TIMEPERIOD',
       'TIME_PERIOD[Code]2', 'TIME_PERIOD2', 'Unnamed: 17', 'Unnamed: 18',
       'Unnamed: 19', 'Unnamed: 20'],
      dtype='object')

In [5]:
detail_df[['OPPO_TIME[CODE]', 'TIME_ON[Code]', 'TIME_ON', 'TIME_PERIOD[Code]',
       'TIME_PERIOD', 'START_TIME']]

,OPPO_TIME[CODE],TIME_ON[Code],TIME_ON,TIME_PERIOD[Code],TIME_PERIOD,START_TIME
0,AM1,AM1,Before 5:00 am,0,OWL,01:00:00
1,AM2,AM2,5:00 am - 6:00 am,1,EARLY AM,05:00:00
2,AM3,AM3,6:00 am - 7:00 am,2,AM,06:00:00
3,MID1,MID1,7:00 am - 8:00 am,2,AM,07:00:00
4,MID2,MID2,8:00 am - 9:00 am,2,AM,08:00:00
5,MID7,MID7,9:00 am - 10:00 am,2,AM,09:00:00
6,MID3,MID3,10:00 am - 11:00 am,3,MID,10:00:00
7,MID4,MID4,11:00 am - 12:00 pm,3,MID,11:00:00
8,MID5,MID5,12:00 pm - 1:00 pm,3,MID,12:00:00
9,MID6,MID6,1:00 pm - 2:00 pm,3,MID,13:00:00


In [6]:
wkday_df.columns

Index(['ROUTE_SURVEYEDCode', 'Route Level Goal', '(0) Goal', '(1) Goal',
       '(2) Goal', '(3) Goal', '(4) Goal', '(5) Goal', '(0) Collect',
       '(1) Collect', '(2) Collect', '(3) Collect', '(4) Collect',
       '(5) Collect', '# of Surveys', '(0) Remain', '(1) Remain', '(2) Remain',
       '(3) Remain', '(4) Remain', '(5) Remain', 'Remaining'],
      dtype='object')

In [7]:
wkday_df[['ROUTE_SURVEYEDCode', 'Route Level Goal', '# of Surveys', 'Remaining']].head()

,ROUTE_SURVEYEDCode,Route Level Goal,# of Surveys,Remaining
0,VTA_2_20,79.625,64,16
1,VTA_2_21,148.875,144,5
2,VTA_2_22,687.500,635,53
3,VTA_2_23,397.750,358,40
4,VTA_2_25,385.000,341,44


In [8]:
wkend_df[['ROUTE_SURVEYEDCode', 'Route Level Goal', '# of Surveys', 'Remaining']].head()

,ROUTE_SURVEYEDCode,Route Level Goal,# of Surveys,Remaining
0,VTA_2_20,2.449599,0,3
1,VTA_2_21,29.853597,10,20
2,VTA_2_22,158.464727,0,159
3,VTA_2_23,96.078731,0,97
4,VTA_2_25,85.034046,0,86


In [13]:
wkend_df['# of Surveys'].sum()

90

In [12]:
wkday_df['# of Surveys'].sum()

6832

In [9]:
wkday_dir_df[['ROUTE_SURVEYEDCode','(0) Goal', '(1) Goal',
       '(2) Goal', '(3) Goal', '(4) Goal', '(5) Goal', '(0) Collect',
       '(1) Collect', '(2) Collect', '(3) Collect', '(4) Collect',
       '(5) Collect', '(0) Remain', '(1) Remain', '(2) Remain',
       '(3) Remain', '(4) Remain', '(5) Remain']].head()

,ROUTE_SURVEYEDCode,(0) Goal,(1) Goal,(2) Goal,(3) Goal,(4) Goal,(5) Goal,(0) Collect,(1) Collect,(2) Collect,(3) Collect,(4) Collect,(5) Collect,(0) Remain,(1) Remain,(2) Remain,(3) Remain,(4) Remain,(5) Remain
0,VTA_2_20_00,0.0,0.0,6.000,15.125,18.375,1.875,1,0,5,13,8,0,0,0,1,3,11,2
1,VTA_2_20_01,0.0,0.0,9.375,16.000,11.250,1.625,0,0,16,16,5,0,0,0,0,0,7,2
2,VTA_2_21_00,0.0,0.0,18.750,32.125,22.250,2.750,0,0,23,43,17,10,0,0,0,0,6,0
3,VTA_2_21_01,0.0,0.0,22.625,24.125,20.625,5.625,1,0,14,24,16,6,0,0,9,1,5,0
4,VTA_2_22_00,13.0,6.0,75.000,124.250,92.375,24.750,0,0,55,130,108,46,13,6,20,0,0,0


In [10]:
wkend_dir_df[['ROUTE_SURVEYEDCode','(0) Goal', '(1) Goal',
       '(2) Goal', '(3) Goal', '(4) Goal', '(5) Goal', '(0) Collect',
       '(1) Collect', '(2) Collect', '(3) Collect', '(4) Collect',
       '(5) Collect', '(0) Remain', '(1) Remain', '(2) Remain',
       '(3) Remain', '(4) Remain', '(5) Remain']].head()

,ROUTE_SURVEYEDCode,(0) Goal,(1) Goal,(2) Goal,(3) Goal,(4) Goal,(5) Goal,(0) Collect,(1) Collect,(2) Collect,(3) Collect,(4) Collect,(5) Collect,(0) Remain,(1) Remain,(2) Remain,(3) Remain,(4) Remain,(5) Remain
0,VTA_2_20_00,0.000000,0.315153,0.415429,0.329478,0.114601,0.000000,1,0,5,13,8,0,0,1,0,0,0,0
1,VTA_2_20_01,0.000000,0.530030,0.329478,0.272178,0.143251,0.000000,0,0,16,16,5,0,0,1,0,0,0,0
2,VTA_2_21_00,0.014325,3.251807,5.615456,5.042450,1.260613,0.000000,0,0,23,43,17,10,1,4,0,0,0,0
3,VTA_2_21_01,0.157577,3.667236,5.386254,4.383494,1.074386,0.000000,1,0,14,24,16,6,0,4,0,0,0,0
4,VTA_2_22_00,1.017085,14.840848,23.736761,23.120780,14.941124,2.048495,0,0,55,130,108,46,2,15,0,0,0,0


In [11]:
# wkend_df.rename(columns={'CR_PRE_Early_AM':'(0) Goal','CR_Early_AM':'(1) Goal','CR_AM_Peak':'(2) Goal','CR_Midday':'(3) Goal','CR_PM_Peak':'(4) Goal','CR_Evening':'(5) Goal',
#          'DB_PRE_Early_AM_Peak':'(0) Collect', 'DB_Early_AM_Peak':'(1) Collect', 'DB_AM_Peak':'(2) Collect',
#        'DB_Midday':'(3) Collect', 'DB_PM_Peak':'(4) Collect', 'DB_Evening':'(5) Collect','PRE_Early_AM_DIFFERENCE':'(0) Remain',
#        'Early_AM_DIFFERENCE':'(1) Remain', 'AM_DIFFERENCE':'(2) Remain', 'Midday_DIFFERENCE':'(3) Remain',
#        'PM_DIFFERENCE':'(4) Remain', 'Evening_DIFFERENCE':'(5) Remain','CR_Overall_Goal':'Route Level Goal','DB_Total':'# of Surveys','Overall_Goal_DIFFERENCE':'Remaining'}).head()